# Dialogue Vectorisation — Turin Walk

This notebook vectorises the 24 Franco–Nietzsche dialogue texts from the Turin walk route.
It was developed as a preliminary vectorisation study (Workshop 4) and later informed the full
cross-dataset vectorisation in the Chapter 2 vectorisation notebook.

Working with only the dialogue texts in isolation allows a more detailed analysis of the
semantic structure of the walk itself — how the language shifts from location to location,
which places cluster together thematically, and what the main semantic oppositions are
across the 24-stop route.

**Methods applied:**
1. TF-IDF — top distinctive terms per location
2. Co-occurrence Word2Vec — semantic neighbours within the dialogue corpus
3. Sentence Embeddings (MiniLM) — dense semantic vectors
4. PCA — Conceptual Map of Turin showing locations in semantic space
5. t-SNE — Semantic neighbourhood clustering
6. Geographic map — top TF-IDF keywords plotted on Turin coordinates

## 1. Setup

In [1]:
!pip install matplotlib pandas scikit-learn sentence-transformers adjustText -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\elena\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [2]:
import os, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

DATA_DIR   = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\APIs\dialogues"
OUTPUT_DIR = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS\dialogues"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 150,
})

C_MAIN = '#1D6A72'
C_ACC  = '#C0462B'

def save(fig, name):
    path = os.path.join(OUTPUT_DIR, name)
    fig.savefig(path, dpi=200, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'Saved: {path}')

print('Setup complete.')

Setup complete.


## 2. Load dialogue texts

Each `.txt` file corresponds to one location on the Turin walk. The files contain the
full dialogue between Franco and Nietzsche at that location.

In [3]:
def load_texts(data_dir):
    paths = sorted(glob.glob(os.path.join(data_dir, '*.txt')))
    records = []
    for p in paths:
        name = Path(p).stem
        with open(p, 'r', encoding='utf-8', errors='ignore') as f:
            txt = f.read()
        records.append({'doc_id': name, 'text': txt})
    return pd.DataFrame(records)

def clean_text(s):
    if pd.isna(s): return ''
    s = re.sub(r'\*\*.*?\*\*', ' ', str(s))
    s = re.sub(r'#+.*?\n', ' ', s)
    s = re.sub(r'\s+', ' ', s)
    return s.strip().lower()

df = load_texts(DATA_DIR)
df['text_clean'] = df['text'].apply(clean_text)

print(f'Loaded {len(df)} dialogue documents')
df[['doc_id']].head()

Loaded 23 dialogue documents


,doc_id
0,Borgo Medievale (Parco del Valen
1,Café Paris
2,Castello del Valentino
3,Cattedrale di San Giovanni Batti
4,Chiesa della Gran Madre di Dio (


## 3. TF-IDF — Linguistic fingerprint per location

TF-IDF identifies the most distinctive words for each location — words that appear
frequently in one dialogue but rarely across the other 23. This reveals the unique
vocabulary of each place as experienced by Franco and Nietzsche.

In [4]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

custom_stop = set(ENGLISH_STOP_WORDS) | {
    'it','its','itself','would','could','might','also','though',
    'nietzsche','franco','turin','torino','nan'
}

vectorizer = TfidfVectorizer(
    stop_words=list(custom_stop),
    ngram_range=(1, 2),
    min_df=2,
    token_pattern=r'(?u)\b[a-zA-ZÀ-ÖØ-öø-ÿ]{3,}\b'
)

X = vectorizer.fit_transform(df['text_clean'])
feature_names = np.array(vectorizer.get_feature_names_out())

def top_terms(row_idx, k=10):
    row = X[row_idx].toarray().ravel()
    top_ids = row.argsort()[-k:][::-1]
    return list(zip(feature_names[top_ids], row[top_ids]))

df['tfidf_top_terms'] = [[t for t, _ in top_terms(i, 10)] for i in range(len(df))]

print(f'TF-IDF matrix: {X.shape}')
df[['doc_id', 'tfidf_top_terms']].head(5)

TF-IDF matrix: (23, 632)


,doc_id,tfidf_top_terms
0,Borgo Medievale (Parco del Valen,"[children, patch, small gathering, benches smo..."
1,Café Paris,"[imagine, local, artisans, invite, café, chair..."
2,Castello del Valentino,"[spirit, invite, scents fresh, mute, speak lab..."
3,Cattedrale di San Giovanni Batti,"[silence, sacred, breath, invite, like, void, ..."
4,Chiesa della Gran Madre di Dio (,"[speak, like, fabric, feels, edges, stone, sto..."


### 3.1 Top TF-IDF terms heatmap across all locations

In [5]:
TOP_K = 8
MAX_COLS = 35

cand_ids = set()
for i in range(X.shape[0]):
    row = X[i].toarray().ravel()
    cand_ids.update(row.argsort()[-TOP_K:])

cand_ids = list(cand_ids)
col_scores = X[:, cand_ids].sum(axis=0).A1
order = np.argsort(-col_scores)
keep_ids = [cand_ids[j] for j in order[:MAX_COLS]]

M = X[:, keep_ids].toarray()
M_norm = M / (M.max(axis=1, keepdims=True) + 1e-9)
terms = feature_names[keep_ids]
docs  = [d.replace('_', ' ')[:28] for d in df['doc_id'].tolist()]

import seaborn as sns
fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(M_norm, ax=ax, cmap='YlOrRd',
            linewidths=0.2, linecolor='white',
            cbar_kws={'label': 'Relative importance', 'shrink': 0.5})
ax.set_xticks(np.arange(len(terms)) + 0.5)
ax.set_xticklabels(terms, rotation=55, ha='right', fontsize=7)
ax.set_yticks(np.arange(len(docs)) + 0.5)
ax.set_yticklabels(docs, fontsize=7.5)
ax.set_title('TF-IDF Heatmap — Linguistic Fingerprint of the Turin Walk',
             fontweight='bold', pad=12)
fig.tight_layout()
save(fig, 'tfidf_heatmap_dialogue.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS\dialogues\tfidf_heatmap_dialogue.png


## 4. Co-occurrence Word2Vec — Semantic neighbours

A co-occurrence matrix is built from the dialogue texts and decomposed with SVD to
produce dense word vectors. This allows querying which words appear in similar contexts
within the Franco–Nietzsche corpus — revealing the semantic structure of the walk vocabulary.

In [6]:
from sklearn.decomposition import TruncatedSVD

def tokenize(text):
    return re.findall(r"[a-zA-ZÀ-ÖØ-öø-ÿ']{3,}", text.lower())

docs_tokens = [tokenize(t) for t in df['text_clean']]
vocab_count = Counter(w for doc in docs_tokens for w in doc)
idx2word = sorted([w for w, c in vocab_count.items() if c >= 2])
word2idx = {w: i for i, w in enumerate(idx2word)}
V = len(idx2word)
print(f'Vocabulary size: {V}')

# Co-occurrence matrix
window = 5
cooc = np.zeros((V, V), dtype=np.float32)
for toks in docs_tokens:
    n = len(toks)
    for i, w in enumerate(toks):
        wi = word2idx.get(w)
        if wi is None: continue
        for j in range(max(0, i-window), min(n, i+window+1)):
            if j == i: continue
            uj = word2idx.get(toks[j])
            if uj is not None:
                cooc[wi, uj] += 1.0

# SVD embeddings
svd = TruncatedSVD(n_components=50, random_state=42)
word_vecs = svd.fit_transform(cooc)
norms = np.linalg.norm(word_vecs, axis=1, keepdims=True) + 1e-8
word_vecs_norm = word_vecs / norms

def similar_words(query, topn=8):
    qi = word2idx.get(query.lower())
    if qi is None: return []
    sims = word_vecs_norm[qi] @ word_vecs_norm.T
    order = np.argsort(-sims)
    return [(idx2word[k], float(sims[k])) for k in order if k != qi][:topn]

query_terms = ['stone','echo','silence','factory','memory','light','body','ruin']
print('\nSemantic neighbours in the dialogue corpus:')
for q in query_terms:
    nbrs = similar_words(q)
    if nbrs:
        print(f'  {q:12} → {[w for w, _ in nbrs[:5]]}')

Vocabulary size: 553

Semantic neighbours in the dialogue corpus:
  stone        → ['river', "city's", 'shadows', 'the', 'pulse']
  echo         → ['amidst', 'industry', 'relentless', 'rhythm', 'corners']
  silence      → ['symmetry', 'still', 'shadows', 'heart', 'facades']
  memory       → ['weight', 'still', 'monument', 'square', 'heart']
  light        → ['dance', 'symphony', 'past', 'scents', 'present']


## 5. Sentence Embeddings — Dense semantic vectors

Each dialogue document is encoded as a 384-dimensional vector using the pre-trained
MiniLM sentence transformer. Documents that share similar atmospheric tone and vocabulary
appear close together in this space regardless of exact word overlap.

In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
emb   = model.encode(df['text_clean'].tolist(), show_progress_bar=True)
print(f'Embeddings shape: {emb.shape}')

np.save(os.path.join(OUTPUT_DIR, 'dialogue_embeddings.npy'), emb)
print('Embeddings saved.')

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (23, 384)
Embeddings saved.


## 6. Conceptual Map of Turin — PCA

PCA projects the 384-dimensional sentence embeddings down to 2 dimensions. The axes
have been interpreted based on the distribution of locations: the horizontal axis
separates urban/conversational spaces from monumental/memorial ones; the vertical axis
separates atmospheric/natural spaces from structured urban geometry.

This map shows the semantic structure of the walk — which locations share similar
language and which are conceptually distant.

In [8]:
pca = PCA(n_components=2, random_state=42)
XY  = pca.fit_transform(emb)

labels = df['doc_id'].tolist()
short  = [(s[:28] + '…') if len(s) > 29 else s for s in labels]
short  = [s.replace('_', ' ') for s in short]

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(XY[:, 0], XY[:, 1], color=C_MAIN, s=55,
           edgecolors='white', linewidth=0.5, zorder=3)

for (x, y, lbl) in zip(XY[:, 0], XY[:, 1], short):
    ax.annotate(lbl, (x, y), fontsize=7.5,
                xytext=(4, 4), textcoords='offset points', color='#333')

ax.axhline(0, color='#ccc', lw=0.7)
ax.axvline(0, color='#ccc', lw=0.7)

ax.set_xlabel('← Urban Life / Conversation     |     Memory / Monumentality →',
              fontsize=10)
ax.set_ylabel('Atmosphere / Nature ↑     |     Urban Geometry ↓',
              fontsize=10)
ax.set_title('Conceptual Map of Turin — PCA of Sentence Embeddings',
             fontsize=13, fontweight='bold', pad=12)

ax.text(0.98, 0.02,
        f'Each dot = one dialogue location\n'
        f'Proximity = similar atmospheric vocabulary\n'
        f'PC1: {pca.explained_variance_ratio_[0]*100:.1f}% | '
        f'PC2: {pca.explained_variance_ratio_[1]*100:.1f}% variance',
        fontsize=8, ha='right', va='bottom', transform=ax.transAxes,
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

fig.tight_layout()
save(fig, 'dialogue_conceptual_map_pca.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS\dialogues\dialogue_conceptual_map_pca.png


## 7. Semantic Neighbourhoods — t-SNE

t-SNE reveals local clustering that PCA may flatten. Locations that cluster
tightly share similar emotional register and vocabulary — they form semantic
neighbourhoods within the walk.

In [9]:
tsne    = TSNE(n_components=2, perplexity=5, metric='cosine',
               random_state=42, max_iter=1000)
XY_tsne = tsne.fit_transform(emb)

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(XY_tsne[:, 0], XY_tsne[:, 1], color=C_ACC, s=55,
           edgecolors='white', linewidth=0.5, zorder=3)

for (x, y, lbl) in zip(XY_tsne[:, 0], XY_tsne[:, 1], short):
    ax.annotate(lbl, (x, y), fontsize=7.5,
                xytext=(4, 4), textcoords='offset points', color='#333')

ax.set_title('Semantic Neighbourhoods of Turin — t-SNE',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Conceptual distance (cosine)', fontsize=10)
ax.set_ylabel('Conceptual distance (cosine)', fontsize=10)

ax.text(0.98, 0.02,
        'Nearby points share similar emotional and linguistic atmosphere.\n'
        't-SNE highlights clusters — the semantic neighbourhoods of the walk.',
        fontsize=8, ha='right', va='bottom', transform=ax.transAxes,
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

fig.tight_layout()
save(fig, 'dialogue_tsne.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS\dialogues\dialogue_tsne.png


## 8. Geographic map — Top TF-IDF keywords on Turin coordinates

The top 3 TF-IDF terms for each location are plotted on the geographic coordinates
of the walk. This connects the semantic analysis back to physical space — showing
where specific vocabularies are anchored in the city.

In [10]:
import unicodedata

locations_data = [
    {'location': 'Statuto (neighborhood arrival)',           'lat': 45.0767, 'lon': 7.6701},
    {'location': 'Piazza Statuto',                           'lat': 45.0767, 'lon': 7.6701},
    {'location': 'Café Paris',                               'lat': 45.0705, 'lon': 7.6868},
    {'location': 'Via Giuseppe Garibaldi, 43-41',            'lat': 45.0731, 'lon': 7.6794},
    {'location': 'Piazza delle Erbe',                        'lat': 45.0682, 'lon': 7.6821},
    {'location': 'Cattedrale di San Giovanni Battista',      'lat': 45.0750, 'lon': 7.6847},
    {'location': 'Porta Palatina',                           'lat': 45.0750, 'lon': 7.6847},
    {'location': 'Palazzo Reale di Torino',                  'lat': 45.0739, 'lon': 7.6870},
    {'location': 'Piazza Castello',                          'lat': 45.0737, 'lon': 7.6865},
    {'location': 'Palazzo Madama',                           'lat': 45.0737, 'lon': 7.6869},
    {'location': 'Palazzo Carignano',                        'lat': 45.0733, 'lon': 7.6854},
    {'location': 'Piazza Carlo Alberto',                     'lat': 45.0739, 'lon': 7.6870},
    {'location': 'Museo Egizio (outside)',                   'lat': 45.0730, 'lon': 7.6799},
    {'location': 'Piazza San Carlo',                         'lat': 45.0673, 'lon': 7.6819},
    {'location': 'Piazza CLN',                               'lat': 45.0683, 'lon': 7.6840},
    {'location': 'Piazza Carlo Emanuele II (Piazza Carlina)','lat': 45.0689, 'lon': 7.6840},
    {'location': 'Giardini Cavour',                          'lat': 45.0646, 'lon': 7.6857},
    {'location': 'Piazza Maria Teresa',                      'lat': 45.0636, 'lon': 7.6835},
    {'location': 'Piazza Vittorio Veneto',                   'lat': 45.0636, 'lon': 7.6890},
    {'location': 'Chiesa della Gran Madre di Dio (steps)',   'lat': 45.0621, 'lon': 7.6985},
    {'location': 'Monumental Arch to the Artillery Force',   'lat': 45.0649, 'lon': 7.6933},
    {'location': 'Castello del Valentino',                   'lat': 45.0569, 'lon': 7.6863},
    {'location': 'Il posto giusto del Baby Luna Park',       'lat': 45.0532, 'lon': 7.6847},
    {'location': 'Borgo Medievale (Parco del Valentino)',    'lat': 45.0486, 'lon': 7.6850},
]
locs = pd.DataFrame(locations_data)

def norm_str(s):
    s = re.sub(r'\(.*?\)', ' ', str(s))
    s = ''.join(c for c in unicodedata.normalize('NFD', s)
                if unicodedata.category(c) != 'Mn')
    return re.sub(r'\s+', ' ', s.lower()).strip()

# Match locations to dialogue files
rows = []
for _, r in locs.iterrows():
    loc_n = norm_str(r['location'])[:12]
    match = df[df['doc_id'].apply(lambda x: norm_str(x)[:12]).str.contains(loc_n[:8])]
    if len(match) > 0:
        kws   = match.iloc[0]['tfidf_top_terms']
        label = ', '.join(kws[:3]) if isinstance(kws, list) else ''
        matched = True
    else:
        label   = ''
        matched = False
    rows.append({**r, 'label': label, 'matched': matched})

merged = pd.DataFrame(rows)
print(f'Matched: {merged["matched"].sum()} / {len(merged)}')

fig, ax = plt.subplots(figsize=(8, 12))
fig.patch.set_facecolor('#fdfcf8')
ax.set_facecolor('#fdfcf8')

# Route line
ax.plot(merged['lon'], merged['lat'], color='#ccc',
        linewidth=0.8, alpha=0.6, zorder=1)

# Dots
colors = [C_MAIN if m else '#ccc' for m in merged['matched']]
ax.scatter(merged['lon'], merged['lat'], c=colors,
           s=40, zorder=3, edgecolors='white', linewidth=0.4)

# Labels
for _, row in merged.iterrows():
    if row['label']:
        ax.text(row['lon'] + 0.0005, row['lat'] + 0.0004,
                f"{row['location'][:20]}\n{row['label']}",
                fontsize=6, color='#444', va='bottom')

ax.set_xlabel('Longitude', fontsize=9)
ax.set_ylabel('Latitude',  fontsize=9)
ax.set_title('Turin Walk — Top TF-IDF Keywords per Location',
             fontweight='bold', pad=12)
ax.grid(True, alpha=0.2, linestyle='--')
fig.tight_layout()
save(fig, 'dialogue_geographic_keyword_map.png')

Matched: 24 / 24
Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS\dialogues\dialogue_geographic_keyword_map.png


## 9. Summary

In [11]:
print('Outputs saved to:', OUTPUT_DIR)
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f'  {f}')

Outputs saved to: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS\dialogues
  dialogue_conceptual_map_pca.png
  dialogue_embeddings.npy
  dialogue_geographic_keyword_map.png
  dialogue_tsne.png
  tfidf_heatmap_dialogue.png
